# W2D3 — 이진화 비교: Global vs Otsu vs Adaptive

## 📌 오늘의 목표
3가지 이진화 방법의 **특성 차이**를 직접 눈으로 확인하고,  
어떤 결함 유형에 어떤 이진화 방식을 선택해야 하는지 판단 기준을 세운다.

## 🎯 배울 함수
| 함수 | 핵심 파라미터 | 특징 |
|------|-------------|------|
| `cv2.threshold` | `thresh`, `type` | 전체 이미지에 동일한 기준값 적용 |
| `cv2.threshold` + `THRESH_OTSU` | 자동 계산 | 히스토그램 분석으로 최적 기준값 자동 결정 |
| `cv2.adaptiveThreshold` | `blockSize`, `C` | 픽셀 주변 영역마다 기준값을 다르게 적용 |

## 📖 원문 자료
> 📖 원리 이해: [LearnOpenCV - Thresholding](https://learnopencv.com/opencv-threshold-python-cpp/)  
> 📋 파라미터 확인: [OpenCV 공식 - Thresholding](https://docs.opencv.org/4.x/d7/d4d/tutorial_py_thresholding.html)

## 🏭 실무 연결
엣지 검출 다음 단계는 **이진화**다.  
이진화는 이미지를 흰색(255)과 검정(0)으로만 만들어  
결함 영역과 배경을 명확히 분리한다.  
이 결과가 이후 윤곽선 검출, 결함 크기 측정의 입력이 된다.

---
## 📦 1. Import & 데이터 로딩

**왜 이진화 전에 blur를 먼저 하나?**  
→ 노이즈 픽셀도 밝기값을 가지므로 이진화 시 엉뚱한 픽셀이 결함으로 잡힌다.  
→ GaussianBlur로 노이즈를 먼저 제거한 뒤 이진화하는 것이 표준 파이프라인이다.

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

SAMPLE_ROOT = Path('../data/sample_images')

img_building = cv2.imread(str(SAMPLE_ROOT / 'building.jpg'), cv2.IMREAD_GRAYSCALE)
img_xray     = cv2.imread(str(SAMPLE_ROOT / 'x-ray.jpg'),   cv2.IMREAD_GRAYSCALE)
img_letter   = cv2.imread(str(SAMPLE_ROOT / 'letter.jpg'),  cv2.IMREAD_GRAYSCALE)

img_building_blur = cv2.GaussianBlur(img_building, (5, 5), 0)
img_xray_blur     = cv2.GaussianBlur(img_xray,     (5, 5), 0)
img_letter_blur   = cv2.GaussianBlur(img_letter,   (5, 5), 0)

print(f'building 크기: {img_building.shape}')
print(f'x-ray    크기: {img_xray.shape}')
print(f'letter   크기: {img_letter.shape}')

---
## 🔧 2. Global Thresholding — 기준값 고정

### cv2.threshold
- **원리**: 픽셀값이 기준값(thresh)보다 크면 maxval, 작으면 0으로 변환
- **파라미터**: `thresh` — 기준값, `maxval` — 밝은 쪽에 줄 값 (보통 255)
- **타입별 동작**:

| 타입 | 동작 |
|------|------|
| `THRESH_BINARY` | 픽셀 > thresh → 255, 나머지 → 0 |
| `THRESH_BINARY_INV` | 픽셀 > thresh → 0, 나머지 → 255 (결함이 밝으면 사용) |
| `THRESH_TRUNC` | 픽셀 > thresh → thresh로 잘림, 나머지 유지 |
| `THRESH_TOZERO` | 픽셀 > thresh → 유지, 나머지 → 0 |

In [ ]:
img = img_building_blur
thresh_val = 127

_, th_binary     = cv2.threshold(img, thresh_val, 255, cv2.THRESH_BINARY)
_, th_binary_inv = cv2.threshold(img, thresh_val, 255, cv2.THRESH_BINARY_INV)
_, th_trunc      = cv2.threshold(img, thresh_val, 255, cv2.THRESH_TRUNC)
_, th_tozero     = cv2.threshold(img, thresh_val, 255, cv2.THRESH_TOZERO)

fig, axes = plt.subplots(1, 5, figsize=(26, 5))
titles = ['원본 (blur 적용)', f'BINARY\n(thresh={thresh_val})', f'BINARY_INV\n(thresh={thresh_val})',
          f'TRUNC\n(thresh={thresh_val})', f'TOZERO\n(thresh={thresh_val})']
images = [img, th_binary, th_binary_inv, th_trunc, th_tozero]

for ax, title, image in zip(axes, titles, images):
    ax.imshow(image, cmap='gray')
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.suptitle('Global Thresholding — 타입별 비교 (Building)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 👁 어디를 봐야 하나
1. **BINARY** — 건물 윤곽이 흰색으로 나오는지, 배경은 검정인지
2. **BINARY_INV** — BINARY와 반전된 결과인지
3. **TRUNC** — 밝은 부분만 잘리고 어두운 부분은 원본 유지인지
4. **TOZERO** — 어두운 부분이 전부 0이 되는지

> 🤔 **판단 질문 1:** 결함이 주변보다 어둡게 나타나는 이미지에서는 BINARY와 BINARY_INV 중 어떤 것을 써야 결함이 흰색으로 잡힐까요?  
> 🤔 **판단 질문 2:** thresh=127로 고정했을 때, 조명이 전체적으로 어두운 이미지에서는 어떤 문제가 생길까요?

---
```
🔨 깨뜨리기 챌린지
- 챌린지 1 [예측형]: thresh를 50으로 낮추면 BINARY 결과가 어떻게 변할까? 예상 후 실행.
- 챌린지 2 [예측형]: thresh를 200으로 높이면 어떻게 될까?
```

#### ✏️ 빈칸 채우기 — 챌린지 1, 2: thresh 값 변화

In [ ]:
# thresh 값을 바꿔서 실행 — ___ 부분을 채우세요
_, th_low  = cv2.threshold(img, ___, 255, cv2.THRESH_BINARY)
_, th_high = cv2.threshold(img, ___, 255, cv2.THRESH_BINARY)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(th_binary, cmap='gray'); axes[0].set_title('기준 (thresh=127)'); axes[0].axis('off')
axes[1].imshow(th_low,    cmap='gray'); axes[1].set_title('thresh 낮춤 (50)');  axes[1].axis('off')
axes[2].imshow(th_high,   cmap='gray'); axes[2].set_title('thresh 높임 (200)'); axes[2].axis('off')
plt.suptitle('챌린지 1, 2 — thresh 값 변화', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔬 3. Otsu's Method — 자동 기준값

**왜 Otsu가 필요하냐?**  
→ Global thresholding은 thresh를 사람이 직접 정해야 한다.  
→ 이미지마다, 조명마다 최적 thresh가 다르기 때문에 매번 수동으로 정하면 실무에서 쓸 수 없다.  
→ Otsu는 히스토그램을 분석해서 **배경과 결함이 가장 잘 나뉘는 기준값을 자동으로 계산**한다.

**어떻게 계산하냐?**  
→ 모든 thresh 후보값(0~255)을 다 시도해보고,  
→ 두 그룹(배경/결함) 내부의 분산이 가장 작아지는 thresh를 선택한다.

In [ ]:
img = img_building_blur

_, th_manual = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
otsu_val, th_otsu = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

print(f'수동 thresh: 127')
print(f'Otsu 자동 계산 thresh: {otsu_val:.0f}')

fig, axes = plt.subplots(1, 4, figsize=(22, 5))

axes[0].imshow(img,        cmap='gray'); axes[0].set_title('원본 (blur)');          axes[0].axis('off')
axes[1].hist(img.ravel(), 256, [0, 256], color='gray')
axes[1].axvline(x=127,       color='blue',  linestyle='--', label=f'수동 (127)')
axes[1].axvline(x=otsu_val,  color='red',   linestyle='--', label=f'Otsu ({otsu_val:.0f})')
axes[1].set_title('히스토그램'); axes[1].legend()
axes[2].imshow(th_manual,  cmap='gray'); axes[2].set_title('수동 thresh=127');       axes[2].axis('off')
axes[3].imshow(th_otsu,    cmap='gray'); axes[3].set_title(f'Otsu (자동={otsu_val:.0f})'); axes[3].axis('off')

plt.suptitle('Otsu — 자동 기준값 vs 수동 기준값', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 👁 어디를 봐야 하나
- **히스토그램** — 픽셀값 분포가 두 봉우리(배경/결함)로 나뉘는지
- **Otsu 기준선** — 두 봉우리 사이에 그어지는지
- **결과 비교** — 수동 127보다 Otsu가 더 깔끔하게 분리되는지

> 🤔 **판단 질문 3:** 히스토그램에 봉우리가 하나만 있는 이미지(균일한 배경)에서 Otsu를 쓰면 어떤 문제가 생길까요?  
> 🤔 **판단 질문 4:** Otsu가 자동으로 thresh를 정해주면 조명 변화에도 강할까요? 왜 그럴까요?

---
```
🔨 깨뜨리기 챌린지
- 챌린지 3 [예측형]: Scratches 이미지에 Otsu를 적용하면 자동 thresh가 Building과 다르게 나올까? 예상 후 실행.
- 챌린지 4 [판단형]: Pitted Surface에 Otsu를 적용했는데 결과가 좋지 않다면 이유는?
```

#### ✏️ 빈칸 채우기 — 챌린지 3: Scratches에 Otsu 적용

In [ ]:
# X-ray 이미지에 Otsu 적용 — ___ 부분을 채우세요
otsu_val_x, th_otsu_x = cv2.threshold(___, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

print(f'X-ray Otsu 자동 thresh: {otsu_val_x:.0f}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(img_xray,      cmap='gray'); axes[0].set_title('원본');                         axes[0].axis('off')
axes[1].imshow(img_xray_blur, cmap='gray'); axes[1].set_title('blur 후');                      axes[1].axis('off')
axes[2].imshow(th_otsu_x,     cmap='gray'); axes[2].set_title(f'Otsu (thresh={otsu_val_x:.0f})'); axes[2].axis('off')
plt.suptitle('챌린지 3 — X-ray Otsu 이진화', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔬 4. Adaptive Thresholding — 영역별 기준값

**왜 Adaptive가 필요하냐?**  
→ 조명이 이미지 전체에서 균일하지 않을 때 Global/Otsu는 실패한다.  
→ 예) 왼쪽은 밝고 오른쪽은 어두운 이미지 → 하나의 thresh로는 전체를 만족할 수 없다.  
→ Adaptive는 각 픽셀 주변 `blockSize × blockSize` 영역의 평균/가중평균을 기준값으로 쓴다.

**파라미터:**
| 파라미터 | 설명 |
|----------|------|
| `blockSize` | 기준값을 계산할 주변 영역 크기 (홀수, 최소 3) |
| `C` | 계산된 평균에서 뺄 상수 — 크면 더 엄격해짐 |
| `ADAPTIVE_THRESH_MEAN_C` | 주변 영역 단순 평균 |
| `ADAPTIVE_THRESH_GAUSSIAN_C` | 주변 영역 가중 평균 (중앙일수록 가중치 높음) |

In [ ]:
img = img_building_blur

_, th_global = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
otsu_val, th_otsu = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

th_adaptive_mean = cv2.adaptiveThreshold(
    img, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, blockSize=11, C=2)

th_adaptive_gauss = cv2.adaptiveThreshold(
    img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, blockSize=11, C=2)

fig, axes = plt.subplots(1, 5, figsize=(28, 5))
titles = ['원본 (blur)', 'Global (127)', f'Otsu ({otsu_val:.0f})', 'Adaptive Mean', 'Adaptive Gaussian']
images = [img, th_global, th_otsu, th_adaptive_mean, th_adaptive_gauss]

for ax, title, image in zip(axes, titles, images):
    ax.imshow(image, cmap='gray')
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.suptitle('이진화 4종 비교 — Building (blockSize=11, C=2)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 👁 어디를 봐야 하나
- **Global vs Otsu** — 수동 127과 자동 Otsu의 결과 차이
- **Adaptive** — 조명이 고르지 않은 영역에서 Global보다 디테일이 살아있는지
- **Mean vs Gaussian** — 경계선이 얼마나 매끄럽게 나오는지

> 🤔 **판단 질문 5:** blockSize를 크게 하면 결과가 어떻게 변할까요? (힌트: 참고하는 영역이 커짐)  
> 🤔 **판단 질문 6:** C 값을 0으로 하면 어떤 효과가 생길까요? 음수로 하면?

---
```
🔨 깨뜨리기 챌린지
- 챌린지 5 [예측형]: blockSize를 3으로 줄이면 Adaptive 결과가 어떻게 변할까?
- 챌린지 6 [예측형]: C를 0으로 바꾸면 어떻게 될까?
- 챌린지 7 [판단형]: 제조 검사에서 Adaptive를 쓰면 좋은 상황과 Global을 쓰는 게 나은 상황은?
```

#### ✏️ 빈칸 채우기 — 챌린지 5: blockSize 변화

In [ ]:
# blockSize를 3으로 줄여서 실행 — ___ 부분을 채우세요
th_small_block = cv2.adaptiveThreshold(
    img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, blockSize=___, C=2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(th_adaptive_gauss, cmap='gray'); axes[0].set_title('Adaptive Gaussian (blockSize=11)'); axes[0].axis('off')
axes[1].imshow(th_small_block,    cmap='gray'); axes[1].set_title('Adaptive Gaussian (blockSize=3)');  axes[1].axis('off')
plt.suptitle('챌린지 5 — blockSize 변화', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔬 5. 결함 유형별 이진화 비교

**왜 결함마다 방법이 달라지냐?**  
→ Scratches는 배경과 명확한 밝기 차이 → Global/Otsu로 충분  
→ Pitted Surface는 미세한 구덩이 → 조명 불균일 가능성 → Adaptive가 유리할 수 있음

In [ ]:
images_to_compare = {
    'X-ray':    (img_xray_blur,   img_xray),
    'Letter':   (img_letter_blur, img_letter),
}

fig, axes = plt.subplots(2, 4, figsize=(24, 10))
col_titles = ['원본', 'Global (BINARY)', 'Otsu (자동)', 'Adaptive Gaussian']

for row_idx, (label, (img_b, img_src)) in enumerate(images_to_compare.items()):
    _, th_g  = cv2.threshold(img_b, 127, 255, cv2.THRESH_BINARY)
    ov, th_o = cv2.threshold(img_b, 0,   255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    th_a     = cv2.adaptiveThreshold(img_b, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                     cv2.THRESH_BINARY, blockSize=11, C=2)

    for col_idx, image in enumerate([img_src, th_g, th_o, th_a]):
        title = col_titles[col_idx]
        if col_idx == 2:
            title = f'Otsu (thresh={ov:.0f})'
        axes[row_idx][col_idx].imshow(image, cmap='gray')
        axes[row_idx][col_idx].set_title(f'{label}\n{title}', fontsize=11)
        axes[row_idx][col_idx].axis('off')

plt.suptitle('이미지 유형별 이진화 비교 — X-ray vs 손글씨', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 👁 어디를 봐야 하나
- **Scratches** — 긁힘 선이 흰색으로 분리되는지, 배경은 깨끗한 검정인지
- **Pitted Surface** — 구덩이 경계가 어떤 방법에서 가장 뚜렷하게 나오는지
- **Adaptive** — 배경 질감이 있는 이미지에서 Global보다 유리한지

> 🤔 **판단 질문 7:** Scratches에서 BINARY_INV를 쓰면 긁힘이 검정으로 나오는데, 이게 유리한 상황은 언제일까요?  
> 🤔 **판단 질문 8:** Pitted Surface에 Adaptive가 오히려 노이즈처럼 보인다면 이유는?

---
```
🔨 깨뜨리기 챌린지
- 챌린지 8 [판단형]: Scratches에 BINARY 대신 BINARY_INV를 쓰면 이후 윤곽선 검출에 영향이 있을까?
- 챌린지 9 [지시형]: "Pitted Surface에서 Adaptive blockSize를 5, 11, 21로 바꿔가며 비교해줘"라고 Claude에게 지시해보세요.
```

#### ✏️ 빈칸 채우기 — 챌린지 9: blockSize 3단계 비교

In [ ]:
# Letter 이미지에 blockSize 변화 — ___ 부분을 채우세요
th_bs5  = cv2.adaptiveThreshold(img_letter_blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, blockSize=___, C=2)
th_bs11 = cv2.adaptiveThreshold(img_letter_blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, blockSize=___, C=2)
th_bs21 = cv2.adaptiveThreshold(img_letter_blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, blockSize=___, C=2)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(th_bs5,  cmap='gray'); axes[0].set_title('blockSize=5');  axes[0].axis('off')
axes[1].imshow(th_bs11, cmap='gray'); axes[1].set_title('blockSize=11'); axes[1].axis('off')
axes[2].imshow(th_bs21, cmap='gray'); axes[2].set_title('blockSize=21'); axes[2].axis('off')
plt.suptitle('챌린지 9 — Letter blockSize 비교', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📝 오늘 배운 것 정리

### 핵심 의사결정 기준
| 상황 | 추천 방법 | 이유 |
|------|----------|------|
| 조명 균일, 배경/결함 명확 | Global + Otsu | 자동으로 최적 thresh 계산 |
| 조명 불균일, 그림자 있음 | Adaptive Gaussian | 영역마다 다른 기준값 적용 |
| 결함이 배경보다 어두움 | BINARY_INV | 결함을 흰색으로 뒤집어서 검출 |
| 히스토그램 봉우리 1개 | Global 직접 설정 | Otsu 자동계산 신뢰 불가 |

### 전체 파이프라인 (W2 종합)
```
원본 이미지
    → GaussianBlur       (W2D1 — 노이즈 제거)
    → 엣지 검출           (W2D2 — Sobel / Canny / Laplacian)
    → 이진화              (W2D3 — Global / Otsu / Adaptive)
    → 윤곽선 검출 (예정)   (W3 — findContours, 결함 영역 추출)
```

### 복습 퀴즈
1. Otsu가 자동으로 thresh를 정하는 원리는?
2. Adaptive Thresholding에서 blockSize와 C의 역할은 각각 무엇인가?
3. BINARY와 BINARY_INV는 언제 각각 쓰나?